# LeetCode 3663 — Find The Least Frequent Digit bằng Python

Link đề: https://leetcode.com/problems/find-the-least-frequent-digit/ · Độ khó: **Easy** · Chủ đề: Hash Table, Math, Counting

Skeleton LeetCode:

```python
class Solution:
    def getLeastFrequentDigit(self, n: int) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong biểu diễn thập phân của `n`, tìm **chữ số xuất hiện ít lần nhất**; nếu nhiều chữ số cùng ít nhất thì lấy **chữ số nhỏ nhất**. Chỉ xét các chữ số **có mặt** trong `n` (tần suất `>= 1`).
2. Đếm tần suất chữ số bằng **mảng 10 ô** (`freq[d]`), bằng `Counter`, và bằng `str.count`, hiểu khi nào dùng cách nào.
3. Chọn phần tử theo **hai tiêu chí** (tần suất nhỏ nhất, rồi chữ số nhỏ nhất) bằng `min` với khóa là **tuple** `(tần suất, chữ số)`.
4. Viết 4 cách (đếm bằng `str.count`, `Counter`, mảng 10 ô quét một lượt, sắp xếp cặp) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: tính cả chữ số **không xuất hiện** (tần suất `0`), phá thế **tie-break** (chọn chữ số lớn), trả về **tần suất** thay vì chữ số, và `most_common()[-1]` không đảm bảo chữ số nhỏ nhất.

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Tìm chữ số xuất hiện **ít lần nhất** trong biểu diễn thập phân của `n`. Nếu có nhiều chữ số cùng tần suất thấp nhất, chọn chữ số **nhỏ nhất** trong số đó. Trả về chữ số đó dưới dạng số nguyên.

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `1553322` | `1` | chữ số `1` xuất hiện **1** lần, các chữ số còn lại (`5, 3, 2`) mỗi chữ số 2 lần |
| 2 | `723344511` | `2` | `7`, `2`, `5` mỗi chữ số xuất hiện **1** lần; nhỏ nhất trong chúng là `2` |

Constraints: `1 <= n <= 2^31 - 1`.

In [1]:
n = 1553322
s = str(n)
freq = {}
for ch in s:
    freq[ch] = freq.get(ch, 0) + 1
print(freq)
best = min(freq.items(), key=lambda kv: (kv[1], int(kv[0])))
print("chữ số ít nhất:", best[0], "với tần suất", best[1])
n = 723344511
print({d: str(n).count(d) for d in sorted(set(str(n)))})

{'1': 1, '5': 2, '3': 2, '2': 2}
chữ số ít nhất: 1 với tần suất 1
{'1': 2, '2': 1, '3': 2, '4': 2, '5': 1, '7': 1}


## 3. Kiến thức nền

### 3.1. Đổi số thành chuỗi chữ số: `str(n)`

`str(n)` cho chuỗi các chữ số (không có dấu vì `n > 0`). Duyệt `for ch in str(n)` lấy từng ký tự; `int(ch)` đổi về số `0..9`. Ngoài ra có thể bóc chữ số bằng `n % 10` và `n //= 10` (không cần chuỗi).

In [2]:
n = 1553322
print(str(n), [int(ch) for ch in str(n)])
x, out = n, []
while x:
    x, d = divmod(x, 10)
    out.append(d)
print(out[::-1])

1553322 [1, 5, 5, 3, 3, 2, 2]
[1, 5, 5, 3, 3, 2, 2]


### 3.2. Mảng đếm 10 ô (`freq`)

Chữ số chỉ có `10` giá trị `0..9` nên ta dùng **mảng cố định** `freq = [0] * 10` và `freq[d] += 1`. Truy cập `O(1)`, không cần dictionary. Đây là kiểu "counting array" cho khóa nhỏ.

In [3]:
freq = [0] * 10
for ch in "1553322":
    freq[int(ch)] += 1
print(freq)                       # chỉ số = chữ số, giá trị = số lần
print([(d, c) for d, c in enumerate(freq) if c > 0])

[0, 1, 2, 2, 0, 2, 0, 0, 0, 0]
[(1, 1), (2, 2), (3, 2), (5, 2)]


### 3.3. `collections.Counter`

`Counter(iterable)` đếm tần suất mỗi phần tử và trả về một dict đặc biệt. Chỉ chứa các khóa **có mặt**, nên "chữ số không xuất hiện" **không** có trong `Counter` (tránh được lỗi tần suất `0`). `most_common()` sắp xếp theo tần suất **giảm dần**.

In [4]:
from collections import Counter
c = Counter("723344511")
print(c)
print(c.most_common())            # tần suất giảm dần; hòa thì giữ thứ tự gặp đầu tiên
print(c["9"], len(c))             # khóa vắng mặt -> 0, nhưng không thuộc c.keys()

Counter({'3': 2, '4': 2, '1': 2, '7': 1, '2': 1, '5': 1})
[('3', 2), ('4', 2), ('1', 2), ('7', 1), ('2', 1), ('5', 1)]
0 6


### 3.4. `min` với khóa là tuple (hai tiêu chí)

Python so sánh tuple **từng phần tử từ trái sang phải**: `(1, 5) < (1, 7) < (2, 0)`. Vậy `min(items, key=lambda kv: (tần_suất, chữ_số))` chọn tần suất nhỏ nhất, **hòa thì chữ số nhỏ nhất** — đúng đề, trong một dòng.

In [5]:
print((1, 5) < (1, 7), (1, 7) < (2, 0))
pairs = [(7, 1), (2, 1), (5, 1), (3, 2), (4, 2)]      # (chữ số, tần suất)
print(min(pairs, key=lambda p: (p[1], p[0])))

True True
(2, 1)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay `n = 723344511`:

```
chữ số theo thứ tự: 7 2 3 3 4 4 5 1 1
đếm: 7->1, 2->1, 3->2, 4->2, 5->1, 1->2
các tần suất có mặt: 1 (7, 2, 5) và 2 (3, 4, 1)
tần suất nhỏ nhất = 1; trong {7, 2, 5} chữ số nhỏ nhất là 2  =>  đáp án 2
```

Ba quy tắc:

1. Chỉ xét chữ số **có mặt** trong `n`. Chữ số `9` (tần suất `0`) **không** được tính là "ít nhất".
2. Tìm tần suất **nhỏ nhất** (trong các tần suất `>= 1`).
3. Trong các chữ số đạt tần suất đó, lấy chữ số **nhỏ nhất**.

Ví dụ biên: `n = 10` -> mỗi chữ số `1`, `0` một lần -> đáp án `0`; `n = 100` -> `1` một lần, `0` hai lần -> `1`.

## 5. Cách 1 — Brute Force / baseline

Với mỗi chữ số `d` từ `0` đến `9`, đếm số lần `d` xuất hiện bằng `str(n).count(str(d))`. Chỉ xét `d` có tần suất `> 0`, giữ lại cặp tốt nhất theo `(tần suất, chữ số)`. Vì duyệt `d` tăng dần và chỉ cập nhật khi tần suất **nhỏ hơn hẳn**, chữ số nhỏ nhất thắng khi hòa.

Nút thắt: mỗi lần `count` quét lại cả chuỗi -> `10` lần quét. Với `<= 10` chữ số thì không đáng kể, nhưng ta đang đếm đi đếm lại cùng dữ liệu.

In [6]:
class SolutionCount:
    def getLeastFrequentDigit(self, n):
        s = str(n)
        best_digit, best_cnt = -1, 10 ** 9
        for d in range(10):
            c = s.count(str(d))
            if c > 0 and c < best_cnt:       # < (không phải <=) để chữ số nhỏ hơn thắng khi hòa
                best_digit, best_cnt = d, c
        return best_digit

print(SolutionCount().getLeastFrequentDigit(1553322))
print(SolutionCount().getLeastFrequentDigit(723344511))

1
2


**Độ phức tạp:** `L` = số chữ số (`<= 10`). Time `O(10 * L)`, Space `O(L)` cho chuỗi.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đếm tất cả tần suất **một lượt** vào mảng `freq[10]`, rồi quét mảng **một lần** để chọn. Chữ số chỉ có `10` giá trị nên bước chọn là hằng số.

**Quy tắc "nếu… thì…":**

- Mỗi chữ số `d` của `n` -> `freq[d] += 1`.
- Quét `d = 0..9`: **nếu** `freq[d] > 0` **và** (`best == -1` **hoặc** `freq[d] < freq[best]`) **thì** `best = d`.
- Vì quét theo thứ tự `d` tăng và dùng so sánh **ngặt** `<`, khi hòa chữ số nhỏ hơn (gặp trước) được giữ lại.

Invariant: sau khi xét `d`, `best` là chữ số nhỏ nhất đạt tần suất nhỏ nhất trong các chữ số `0..d` có mặt.

In [7]:
def least_digit(n):
    freq = [0] * 10
    while n:
        n, d = divmod(n, 10)
        freq[d] += 1
    best = -1
    for d in range(10):
        if freq[d] > 0 and (best == -1 or freq[d] < freq[best]):
            best = d
    return best

print(least_digit(1553322), least_digit(723344511), least_digit(10), least_digit(100))

1 2 0 1


## 7. Các cách tiếp theo

### Cách 2 — `Counter` + `min` với khóa tuple

`Counter` chỉ chứa chữ số có mặt. Chọn `min` theo `(tần suất, chữ số)`: đúng hai tiêu chí của đề trong một dòng. Ngắn nhất, đọc gần như đề bài.

In [8]:
from collections import Counter

class SolutionCounter:
    def getLeastFrequentDigit(self, n):
        cnt = Counter(int(ch) for ch in str(n))
        return min(cnt, key=lambda d: (cnt[d], d))

print(SolutionCounter().getLeastFrequentDigit(1553322), SolutionCounter().getLeastFrequentDigit(723344511))

1 2


**Độ phức tạp:** Time `O(L)`, Space `O(1)` (tối đa 10 khóa).

### Cách 3 — Mảng 10 ô, bóc chữ số bằng `divmod`, quét một lượt (cách nộp bài)

Không đổi sang chuỗi, không import. Đếm vào `freq`, quét `0..9` giữ chữ số tốt nhất bằng `<` ngặt.

In [9]:
class SolutionArray:
    def getLeastFrequentDigit(self, n):
        freq = [0] * 10
        while n > 0:
            n, d = divmod(n, 10)
            freq[d] += 1
        best = -1
        for d in range(10):
            if freq[d] > 0 and (best == -1 or freq[d] < freq[best]):
                best = d
        return best

print(SolutionArray().getLeastFrequentDigit(1553322), SolutionArray().getLeastFrequentDigit(723344511))

1 2


**Độ phức tạp:** Time `O(L + 10)`, Space `O(1)` (mảng 10 ô).

### Cách 4 — Sắp xếp các cặp `(tần suất, chữ số)`

Tạo danh sách cặp `(freq[d], d)` cho các `d` có mặt, sắp xếp tăng dần rồi lấy phần tử đầu. Sắp xếp tuple cũng so sánh trái sang phải nên tie-break ra chữ số nhỏ nhất. Hơi thừa (sắp xếp thay vì `min`), nhưng cho thấy "sắp xếp theo nhiều khóa" là kỹ thuật chung.

In [10]:
class SolutionSort:
    def getLeastFrequentDigit(self, n):
        s = str(n)
        pairs = sorted((s.count(ch), int(ch)) for ch in set(s))
        return pairs[0][1]

print(SolutionSort().getLeastFrequentDigit(1553322), SolutionSort().getLeastFrequentDigit(723344511))

1 2


**Độ phức tạp:** Time `O(L + k log k)` với `k <= 10` là số chữ số khác nhau, Space `O(k)`.

## 8. TRACE từng bước

Trace `n = 723344511` với mảng `freq`. Lần lượt bóc chữ số từ phải sang trái: `1, 1, 5, 4, 4, 3, 3, 2, 7`.

| Bước | chữ số d | freq (chỉ các ô > 0) |
|---|---|---|
| 1 | 1 | `{1: 1}` |
| 2 | 1 | `{1: 2}` |
| 3 | 5 | `{1: 2, 5: 1}` |
| 4-5 | 4, 4 | `{1: 2, 4: 2, 5: 1}` |
| 6-7 | 3, 3 | `{1: 2, 3: 2, 4: 2, 5: 1}` |
| 8 | 2 | `{..., 2: 1}` |
| 9 | 7 | `{..., 7: 1}` |

Quét `d = 0..9`: `d=1` (cnt 2) -> `best=1`; `d=2` (cnt 1 < 2) -> `best=2`; `d=3,4` (cnt 2) không tốt hơn; `d=5` (cnt 1 **không nhỏ hơn** 1) giữ `2`; `d=7` giữ `2`. Kết quả `2`.

In [11]:
def trace_least(n):
    print("n =", n)
    freq = [0] * 10
    x = n
    while x:
        x, d = divmod(x, 10)
        freq[d] += 1
        print(f"  bóc chữ số {d}: freq={ {i: c for i, c in enumerate(freq) if c} }")
    best = -1
    for d in range(10):
        if freq[d] == 0:
            continue
        if best == -1 or freq[d] < freq[best]:
            print(f"  d={d}: freq={freq[d]} tốt hơn best={best} -> cập nhật")
            best = d
        else:
            print(f"  d={d}: freq={freq[d]} không nhỏ hơn freq[best]={freq[best]} -> giữ best={best}")
    print("  => đáp án", best)
    return best

trace_least(723344511)
trace_least(1553322)

n = 723344511
  bóc chữ số 1: freq={1: 1}
  bóc chữ số 1: freq={1: 2}
  bóc chữ số 5: freq={1: 2, 5: 1}
  bóc chữ số 4: freq={1: 2, 4: 1, 5: 1}
  bóc chữ số 4: freq={1: 2, 4: 2, 5: 1}
  bóc chữ số 3: freq={1: 2, 3: 1, 4: 2, 5: 1}
  bóc chữ số 3: freq={1: 2, 3: 2, 4: 2, 5: 1}
  bóc chữ số 2: freq={1: 2, 2: 1, 3: 2, 4: 2, 5: 1}
  bóc chữ số 7: freq={1: 2, 2: 1, 3: 2, 4: 2, 5: 1, 7: 1}
  d=1: freq=2 tốt hơn best=-1 -> cập nhật
  d=2: freq=1 tốt hơn best=1 -> cập nhật
  d=3: freq=2 không nhỏ hơn freq[best]=1 -> giữ best=2
  d=4: freq=2 không nhỏ hơn freq[best]=1 -> giữ best=2
  d=5: freq=1 không nhỏ hơn freq[best]=1 -> giữ best=2
  d=7: freq=1 không nhỏ hơn freq[best]=1 -> giữ best=2
  => đáp án 2
n = 1553322
  bóc chữ số 2: freq={2: 1}
  bóc chữ số 2: freq={2: 2}
  bóc chữ số 3: freq={2: 2, 3: 1}
  bóc chữ số 3: freq={2: 2, 3: 2}
  bóc chữ số 5: freq={2: 2, 3: 2, 5: 1}
  bóc chữ số 5: freq={2: 2, 3: 2, 5: 2}
  bóc chữ số 1: freq={1: 1, 2: 2, 3: 2, 5: 2}
  d=1: freq=1 tốt hơn best=-1 -> cậ

1

## 9. Code nộp LeetCode

In [12]:
from typing import List, Optional

class Solution:
    def getLeastFrequentDigit(self, n: int) -> int:
        freq = [0] * 10
        while n > 0:
            n, d = divmod(n, 10)
            freq[d] += 1
        best = -1
        for d in range(10):
            if freq[d] > 0 and (best == -1 or freq[d] < freq[best]):
                best = d
        return best

## 10. Vì sao thuật toán đúng

- **Đếm đúng:** `divmod(n, 10)` bóc từng chữ số đúng một lần, nên `freq[d]` bằng chính xác số lần `d` xuất hiện.
- **Chỉ xét chữ số có mặt:** điều kiện `freq[d] > 0` loại các chữ số vắng mặt. `n >= 1` nên có ít nhất một chữ số có mặt, vì vậy `best` luôn được gán (không còn `-1`).
- **Chọn đúng tần suất nhỏ nhất:** theo invariant, sau mỗi `d` thì `best` có tần suất nhỏ nhất trong `0..d`; kết thúc ở `d = 9` là nhỏ nhất toàn cục.
- **Tie-break:** ta duyệt `d` **tăng dần** và chỉ cập nhật khi `freq[d] < freq[best]` (ngặt). Khi hòa, chữ số đã được chọn trước đó (nhỏ hơn) được giữ nguyên, nên kết quả là chữ số nhỏ nhất trong nhóm.
- **Không phụ thuộc thứ tự chữ số trong `n`:** chỉ phụ thuộc tần suất và giá trị chữ số.

## 11. Phân tích Time / Space Complexity

`L` = số chữ số của `n` (`L <= 10` vì `n <= 2^31 - 1 = 2147483647`).

| Cách | Time | Space |
|---|---|---|
| `str.count` cho từng chữ số | `O(10 * L)` | `O(L)` |
| `Counter` + `min` | `O(L)` | `O(1)` |
| Mảng 10 ô + quét | `O(L + 10)` | `O(1)` |
| Sắp xếp cặp | `O(L + k log k)` | `O(k)` |

Mọi cách đều tức thì với `L <= 10`. Cách mảng 10 ô là chuẩn: không import, bộ nhớ hằng số.

## 12. Bộ test

Ví dụ của đề, biên (`n = 1`, `n = 2^31 - 1`), các ca hòa tần suất, rồi đối chiếu **tất cả cách** trên nhiều `n`.

In [13]:
impls = [SolutionCount(), SolutionCounter(), SolutionArray(), SolutionSort(), Solution()]

cases = [
    (1553322, 1), (723344511, 2), (1, 1),
    (10, 0),             # 0 và 1 đều 1 lần -> chữ số nhỏ nhất là 0
    (100, 1),            # 1 một lần, 0 hai lần
    (2147483647, 1),     # biên trên của đề
    (11, 1), (121, 2), (112233, 1), (990, 0),
]
for n, exp in cases:
    assert 1 <= n <= 2 ** 31 - 1       # tôn trọng constraints của đề
    for impl in impls:
        got = impl.getLeastFrequentDigit(n)
        assert got == exp, (type(impl).__name__, n, got, exp)
print("ví dụ + edge OK")

ví dụ + edge OK


Kiểm tra tay vài ca: `2147483647` có `2:1, 1:1, 4:3, 7:2, 8:1, 3:1, 6:1` -> tần suất nhỏ nhất `1` thuộc `{1, 2, 3, 6, 8}`, nhỏ nhất là `1`. `990` có `9:2, 0:1` -> `0`. `121` có `1:2, 2:1` -> `2`.

In [14]:
import random
random.seed(3663)

def ref(n):
    cnt = Counter(str(n))
    m = min(cnt.values())
    return min(int(d) for d, c in cnt.items() if c == m)

for _ in range(2000):
    n = random.randint(1, 2 ** 31 - 1)
    exp = ref(n)
    for impl in impls:
        assert impl.getLeastFrequentDigit(n) == exp, (n, type(impl).__name__)
print("2000 test ngẫu nhiên (seed=3663) OK")
for L in range(1, 10):                      # nhiều chữ số lặp: chữ số lấy từ bảng nhỏ
    for _ in range(100):
        first = random.choice("123")            # chữ số đầu khác 0
        n = int(first + "".join(random.choice("0123") for _ in range(L - 1)))
        assert 1 <= n <= 2 ** 31 - 1
        assert Solution().getLeastFrequentDigit(n) == ref(n)
print("test nhiều chữ số lặp OK")

2000 test ngẫu nhiên (seed=3663) OK
test nhiều chữ số lặp OK


In [15]:
# Tính chất: đáp án luôn là chữ số có mặt trong n
for _ in range(500):
    n = random.randint(1, 2 ** 31 - 1)
    ans = Solution().getLeastFrequentDigit(n)
    assert str(ans) in str(n)
# Hoán vị chữ số không đổi đáp án
for _ in range(200):
    n = random.randint(10, 10 ** 9)
    digits = list(str(n))
    random.shuffle(digits)
    m = int("".join(digits).lstrip("0") or "1")
    if sorted(str(m)) == sorted(str(n)):
        assert Solution().getLeastFrequentDigit(n) == Solution().getLeastFrequentDigit(m)
print("tính chất: đáp án có mặt trong n và bất biến khi hoán vị chữ số OK")

tính chất: đáp án có mặt trong n và bất biến khi hoán vị chữ số OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: tính cả chữ số không xuất hiện

Nếu lấy `min` trên toàn bộ mảng `freq[0..9]` thì chữ số vắng mặt có tần suất `0` luôn thắng.

In [16]:
def wrong_include_zero_count(n):
    freq = [0] * 10
    for ch in str(n):
        freq[int(ch)] += 1
    return freq.index(min(freq))          # SAI: min(freq) có thể là 0 (chữ số vắng mặt)

print(wrong_include_zero_count(1553322), "!=", Solution().getLeastFrequentDigit(1553322))
assert wrong_include_zero_count(1553322) != Solution().getLeastFrequentDigit(1553322)   # 0 != 1

0 != 1


### Lỗi 2: tie-break sai (chọn chữ số lớn)

Dùng `<=` khi quét tăng dần sẽ **ghi đè** bằng chữ số lớn hơn khi hòa.

In [17]:
def wrong_tie_largest(n):
    freq = [0] * 10
    for ch in str(n):
        freq[int(ch)] += 1
    best = -1
    for d in range(10):
        if freq[d] > 0 and (best == -1 or freq[d] <= freq[best]):    # SAI: <= thay vì <
            best = d
    return best

print(wrong_tie_largest(723344511), "!=", Solution().getLeastFrequentDigit(723344511))
assert wrong_tie_largest(723344511) != Solution().getLeastFrequentDigit(723344511)      # 7 != 2

7 != 2


### Lỗi 3: trả về tần suất thay vì chữ số

Đề hỏi **chữ số**, không phải số lần xuất hiện nhỏ nhất.

In [18]:
def wrong_return_freq(n):
    cnt = Counter(str(n))
    return min(cnt.values())              # SAI: trả tần suất

print(wrong_return_freq(723344511), "!=", Solution().getLeastFrequentDigit(723344511))
assert wrong_return_freq(723344511) != Solution().getLeastFrequentDigit(723344511)      # 1 != 2

1 != 2


### Lỗi 4: dùng `most_common()[-1]`

`most_common()` sắp xếp theo tần suất giảm dần; khi hòa nó giữ **thứ tự gặp đầu tiên**, nên phần tử cuối không phải chữ số nhỏ nhất mà là chữ số xuất hiện **muộn nhất** trong nhóm hòa.

In [19]:
def wrong_most_common(n):
    return int(Counter(str(n)).most_common()[-1][0])

print(wrong_most_common(723344511), "!=", Solution().getLeastFrequentDigit(723344511))
assert wrong_most_common(723344511) != Solution().getLeastFrequentDigit(723344511)      # 5 != 2

5 != 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chữ số/ký tự xuất hiện ít nhất / nhiều nhất, hòa thì lấy nhỏ nhất" -> **mảng đếm + quét theo thứ tự khóa** hoặc `min(key=(tần_suất, khóa))`.
- Khóa nhỏ cố định (`10` chữ số, `26` chữ cái) -> **mảng đếm** thay cho dictionary.
- Hòa tần suất -> nghĩ ngay đến **tuple** làm khóa so sánh hoặc so sánh ngặt khi quét tăng dần.
- Luôn tự hỏi: có tính khóa **vắng mặt** (tần suất `0`) không? Đề thường nói rõ chỉ xét phần tử có mặt.
- `n` tối đa `2^31 - 1` (10 chữ số) nên không lo hiệu năng; chú ý đọc kỹ đề thay vì tối ưu.
- Đừng dùng `most_common()` cho tie-break: thứ tự khi hòa không phải thứ tự khóa.

## 15. Biến thể của bài

1. Tìm chữ số **xuất hiện nhiều nhất** (hòa -> nhỏ nhất hoặc lớn nhất).
2. **First Unique Character in a String** (LeetCode 387): ký tự đầu tiên có tần suất `1`.
3. **Sort Characters By Frequency** (451): sắp xếp theo tần suất giảm dần.
4. Trả về **tất cả** chữ số đạt tần suất ít nhất (không chỉ chữ số nhỏ nhất).
5. Cơ số khác 10 (hệ nhị phân, thập lục phân): đổi mảng `freq` có kích thước bằng cơ số.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| `str.count` | đếm từng `d` bằng `count` | `O(10 L)` | `O(L)` | Viết nhanh, dễ hiểu |
| `Counter` + `min` | khóa `(tần_suất, chữ_số)` | `O(L)` | `O(1)` | Gọn nhất, đọc gần như đề |
| Mảng 10 ô + quét | `freq[d]`, quét `<` ngặt | `O(L + 10)` | `O(1)` | Cách nộp chuẩn, không import |
| Sắp xếp cặp | `sorted` rồi lấy đầu | `O(L + k log k)` | `O(k)` | Khi cần cả bảng xếp hạng |

Bài nhỏ nên chọn cách bạn viết đúng ngay lần đầu; mảng 10 ô là lựa chọn an toàn.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().getLeastFrequentDigit(2233)` là gì?

**Bài 2 (tính tay):** `n = 5551233`: đếm tần suất và tìm đáp án.

**Bài 3 (điền TODO):** hoàn thành `count_digits(n)` trả mảng `freq` 10 ô.

**Bài 4 (tự viết lại):** viết `most_frequent_digit(n)` trả chữ số xuất hiện **nhiều nhất**, hòa thì chữ số **nhỏ nhất**.

**Bài 5 (mini challenge):** viết `all_least_frequent(n)` trả **danh sách tăng dần** mọi chữ số đạt tần suất ít nhất.

In [20]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().getLeastFrequentDigit(2233))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_digits_todo(n):
    freq = [0] * 10
    # TODO: bóc từng chữ số bằng divmod(n, 10) rồi tăng freq[d]
    return freq

2


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `2` và `3` mỗi chữ số hai lần -> hòa. Bài 2: `5` xuất hiện 3 lần, `3` hai lần, `1` và `2` mỗi chữ số một lần.

**Mức 2 (vừa):** Bài 3: vòng `while n:` với `n, d = divmod(n, 10)`. Bài 4: giống mảng 10 ô nhưng so sánh `>` ngặt. Bài 5: tìm `m = min` các `freq[d] > 0`, rồi liệt kê `d` có `freq[d] == m`.

**Mức 3 (gần đáp án):** Bài 1: hòa tần suất `2` -> chữ số nhỏ nhất `2`. Bài 2: tần suất nhỏ nhất là `1` ở `{1, 2}` -> `1`. Bài 4: quét `d` tăng dần, cập nhật khi `freq[d] > freq[best]`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1
assert Solution().getLeastFrequentDigit(2233) == 2

# Bài 2: 5551233 -> 5:3, 1:1, 2:1, 3:2 -> tần suất nhỏ nhất 1 ở {1, 2} -> 1
assert Solution().getLeastFrequentDigit(5551233) == 1

# Bài 3
def count_digits(n):
    freq = [0] * 10
    while n > 0:
        n, d = divmod(n, 10)
        freq[d] += 1
    return freq
assert count_digits(1553322) == [0, 1, 2, 2, 0, 2, 0, 0, 0, 0]
assert sum(count_digits(723344511)) == 9

# Bài 4
def most_frequent_digit(n):
    freq = count_digits(n)
    best = -1
    for d in range(10):
        if freq[d] > 0 and (best == -1 or freq[d] > freq[best]):
            best = d
    return best
assert most_frequent_digit(723344511) == 1          # 1, 3, 4 đều 2 lần -> nhỏ nhất là 1
assert most_frequent_digit(5551233) == 5
assert most_frequent_digit(7) == 7

# Bài 5
def all_least_frequent(n):
    freq = count_digits(n)
    m = min(c for c in freq if c > 0)
    return [d for d in range(10) if freq[d] == m]
assert all_least_frequent(723344511) == [2, 5, 7]
assert all_least_frequent(1553322) == [1]
assert all_least_frequent(2233) == [2, 3]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tần suất ít nhất, hòa thì khóa nhỏ nhất":**

```python
freq = [0] * 10
while n > 0:
    n, d = divmod(n, 10)
    freq[d] += 1
best = -1
for d in range(10):                                   # quét khóa tăng dần
    if freq[d] > 0 and (best == -1 or freq[d] < freq[best]):   # < ngặt: hòa giữ khóa nhỏ
        best = d
return best
```

**Checklist trước khi submit:**

- [ ] Chỉ xét chữ số **có mặt** (`freq[d] > 0`), không tính tần suất `0`?
- [ ] Hòa tần suất -> lấy chữ số **nhỏ nhất** (quét tăng dần + so sánh ngặt `<`)?
- [ ] Trả về **chữ số**, không phải tần suất?
- [ ] Đã thử `n = 1`, `n = 10` (đáp án `0`), `n = 100`, `2147483647`?
- [ ] Không dùng `most_common()[-1]` cho tie-break?
- [ ] Chữ ký đúng `getLeastFrequentDigit(self, n: int) -> int`?